# Qwen3-1.7B FFT — Notebook 1: EDA + data prep

**Цель:** подготовить subset 50K из `allenai/tulu-3-sft-mixture` для full fine-tune Qwen3-1.7B-base → instruct.

**Выход:** Kaggle Dataset `kotshubin/qwen-fft-data-50k` с `train.jsonl` (47 500), `val.jsonl` (2 500) и `dataset_stats.json`.

**Правила отбора:**
- Только примеры, у которых после применения ChatML-шаблона `total_tokens <= 1024` (strict drop, не truncate).
- `num_turns <= 4` (ограничение длины).
- MD5-дедупликация по конкатенации всех `content`.
- Stratified sample по `source` (сохраняем пропорции микса).
- Train/val split 95/5 стратифицированно.

## 1. Setup

In [1]:
!pip install -q -U datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 64.1 MB/s eta 0:00:00


In [2]:
import hashlib
import json
import os
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

from kaggle_secrets import UserSecretsClient

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_token_super")

MODEL_NAME = "Qwen/Qwen3-1.7B"
DATA_NAME = "allenai/tulu-3-sft-mixture"
MAX_SEQ_LEN = 1024
MAX_TURNS = 4
TARGET_SIZE = 50_000
VAL_FRACTION = 0.05

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

# Qwen3-base уже имеет <|im_start|>, <|im_end|> в словаре — проверяем одним токеном
for special in ["<|im_start|>", "<|im_end|>"]:
    ids = tok.encode(special, add_special_tokens=False)
    assert len(ids) == 1, f"{special} токенизируется в {ids}, ожидали один токен"

print("vocab_size:", tok.vocab_size)
print("eos/pad/bos:", tok.eos_token, tok.pad_token, tok.bos_token)
print("<|im_start|> id:", tok.encode("<|im_start|>", add_special_tokens=False))
print("<|im_end|> id:", tok.encode("<|im_end|>", add_special_tokens=False))

# У Qwen-base chat_template может быть None. Задаём ChatML вручную.
CHATML_TEMPLATE = (
    "{% for m in messages %}"
    "<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)
tok.chat_template = CHATML_TEMPLATE

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

vocab_size: 151643
eos/pad/bos: <|im_end|> <|endoftext|> None
<|im_start|> id: [151644]
<|im_end|> id: [151645]


## 2. Загрузка Tulu-3 SFT mixture

In [4]:
ds = load_dataset(DATA_NAME, split="train")
print("Всего примеров:", len(ds))
print("Колонки:", ds.column_names)
print("Пример:")
print(json.dumps(ds[0], indent=2, ensure_ascii=False)[:800])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/361M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/477M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/147M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/162M [00:00<?, ?B/s]

data/train-00004-of-00006.parquet:   0%|          | 0.00/150M [00:00<?, ?B/s]

data/train-00005-of-00006.parquet:   0%|          | 0.00/116M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/939343 [00:00<?, ? examples/s]

Всего примеров: 939343
Колонки: ['id', 'messages', 'source']
Пример:
{
  "id": "oasst1_5921",
  "messages": [
    {
      "content": "Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.",
      "role": "user"
    },
    {
      "content": "Sure, here's an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:\n``` \n# Configure the AWS provider\nprovider \"aws\" {\n  region = \"us-east-1\"\n}\n\n# Create a security group to allow traffic to the ALB\nresource \"aws_security_group\" \"alb_sg\" {\n  name_prefix = \"alb_sg\"\n  ingress {\n    from_port = 80\n    to_port = 80\n    protocol = \"tcp\"\n    cidr_blocks = [\"0.0.0.0/0\"]\n  }\n}\n\n# Create an ALB and target group\nresource \"aws_lb\"


## 3. EDA

Смотрим распределение по `source`, длины (в токенах) и долю примеров, которые не влезут в 1024.

In [5]:
source_counts = Counter(ds["source"])
print("Источников:", len(source_counts))
for src, cnt in source_counts.most_common():
    print(f"  {cnt:>7}  {src}")

Источников: 19
   149960  ai2-adapt-dev/personahub_math_v5_regen_149960
   107276  ai2-adapt-dev/evol_codealpaca_heval_decontaminated
   100000  ai2-adapt-dev/tulu_v3.9_wildchat_100k
   100000  ai2-adapt-dev/tulu_v3.9_aya_100k
    89982  ai2-adapt-dev/flan_v2_converted
    64312  ai2-adapt-dev/numinamath_tir_math_decontaminated
    50000  ai2-adapt-dev/tulu_v3.9_open_math_2_gsm8k_50k
    50000  ai2-adapt-dev/tulu_v3.9_wildjailbreak_decontaminated_50k
    50000  ai2-adapt-dev/tulu_v3.9_synthetic_finalresp_wildguardmixtrain_decontaminated_50k
    49980  allenai/tulu-3-sft-personas-math-grade
    34999  ai2-adapt-dev/personahub_code_v2_34999
    29980  ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980
    20000  ai2-adapt-dev/tulu_v3.9_personahub_math_interm_algebra_20k
    10983  ai2-adapt-dev/coconot_converted
    10000  ai2-adapt-dev/tulu_v3.9_sciriff_10k
     9500  ai2-adapt-dev/no_robots_converted
     7131  ai2-adapt-dev/oasst1_converted
     5000  ai2-adapt-dev/tulu_v3.9_table_g

In [6]:
# Длина в токенах после применения chat template.
# Считаем на рандомной выборке 20K — полное токенизация 939K долгая и нам не нужна
# для перцентилей.

SAMPLE_FOR_STATS = 20_000
stat_idx = random.sample(range(len(ds)), min(SAMPLE_FOR_STATS, len(ds)))

lengths = []
num_turns_list = []
lens_by_source = defaultdict(list)

for i in stat_idx:
    ex = ds[i]
    messages = ex["messages"]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    n_tok = len(tok.encode(text, add_special_tokens=False))
    lengths.append(n_tok)
    num_turns_list.append(len(messages))
    lens_by_source[ex["source"]].append(n_tok)

lengths = np.array(lengths)
print("\nДлина (токены):")
for p in [50, 75, 90, 95, 99]:
    print(f"  p{p}: {np.percentile(lengths, p):.0f}")
print(f"  max: {lengths.max()}")
print(f"  доля > {MAX_SEQ_LEN}: {(lengths > MAX_SEQ_LEN).mean():.2%}")

print("\nКоличество сообщений в диалоге (turns):")
turns = Counter(num_turns_list)
for k in sorted(turns):
    print(f"  {k} сообщений: {turns[k]}")


Длина (токены):
  p50: 446
  p75: 957
  p90: 1500
  p95: 2029
  p99: 6176
  max: 79624
  доля > 1024: 22.85%

Количество сообщений в диалоге (turns):
  2 сообщений: 19071
  4 сообщений: 349
  6 сообщений: 173
  7 сообщений: 12
  8 сообщений: 100
  9 сообщений: 4
  10 сообщений: 72
  11 сообщений: 1
  12 сообщений: 55
  14 сообщений: 39
  16 сообщений: 19
  18 сообщений: 9
  20 сообщений: 14
  22 сообщений: 8
  24 сообщений: 12
  26 сообщений: 7
  28 сообщений: 11
  30 сообщений: 9
  32 сообщений: 2
  34 сообщений: 2
  36 сообщений: 3
  38 сообщений: 5
  40 сообщений: 4
  42 сообщений: 4
  46 сообщений: 1
  48 сообщений: 1
  50 сообщений: 1
  52 сообщений: 1
  54 сообщений: 3
  56 сообщений: 1
  58 сообщений: 1
  60 сообщений: 1
  64 сообщений: 1
  68 сообщений: 1
  72 сообщений: 1
  78 сообщений: 1
  96 сообщений: 1


In [7]:
# Для каждого источника — p50, p95 и доля > MAX_SEQ_LEN,
# чтобы понимать, из каких источников больше всего теряем при strict drop.

print(f"{'source':<40} {'n':>6} {'p50':>6} {'p95':>6} {'>1024':>8}")
for src in sorted(lens_by_source):
    arr = np.array(lens_by_source[src])
    print(
        f"{src:<40} {len(arr):>6} {np.percentile(arr, 50):>6.0f} "
        f"{np.percentile(arr, 95):>6.0f} {(arr > MAX_SEQ_LEN).mean():>7.1%}"
    )

source                                        n    p50    p95    >1024
ai2-adapt-dev/coconot_converted             272    103    131    0.4%
ai2-adapt-dev/evol_codealpaca_heval_decontaminated   2310    471   1060    5.6%
ai2-adapt-dev/flan_v2_converted            1863    221    915    3.3%
ai2-adapt-dev/no_robots_converted           223    235    614    1.3%
ai2-adapt-dev/numinamath_tir_math_decontaminated   1401    600   1536   16.8%
ai2-adapt-dev/oasst1_converted              136    330   1034    5.9%
ai2-adapt-dev/personahub_code_v2_34999      718    348    729    0.3%
ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980    679    356    904    4.0%
ai2-adapt-dev/personahub_math_v5_regen_149960   3142   1281   2091   79.7%
ai2-adapt-dev/tulu_hard_coded_repeated_10      3     30    110    0.0%
ai2-adapt-dev/tulu_v3.9_aya_100k           2034    146   1965   11.2%
ai2-adapt-dev/tulu_v3.9_open_math_2_gsm8k_50k   1086    221    379    0.0%
ai2-adapt-dev/tulu_v3.9_personahub_math_interm_a

## 4. Фильтрация и сэмплинг 50K

**Шаги:**
1. Сохраняем только примеры с `num_turns <= 4`.
2. Токенизируем chat-template для каждого кандидата; оставляем только `<= 1024`.
3. Дедупликация по MD5 конкатенации `role:content` всех сообщений.
4. Stratified sampling 50K по `source` с сохранением пропорций (после фильтрации).

In [8]:
# Шаг 1: фильтр по числу turns.
n_before = len(ds)
ds = ds.filter(lambda ex: len(ex["messages"]) <= MAX_TURNS, num_proc=4)
print(f"После фильтра num_turns <= {MAX_TURNS}: {len(ds)} / {n_before}")

Filter (num_proc=4):   0%|          | 0/939343 [00:00<?, ? examples/s]

После фильтра num_turns <= 4: 913277 / 939343


In [9]:
# Шаг 2: токенизация + фильтр по длине.
# Делаем через .map с батчами — быстрее, чем .filter c медленной функцией.

def add_token_len(batch):
    lens = []
    for messages in batch["messages"]:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        lens.append(len(tok.encode(text, add_special_tokens=False)))
    return {"token_len": lens}

ds = ds.map(add_token_len, batched=True, batch_size=1000, num_proc=4)
n_before = len(ds)
ds = ds.filter(lambda ex: ex["token_len"] <= MAX_SEQ_LEN, num_proc=4)
print(f"После фильтра token_len <= {MAX_SEQ_LEN}: {len(ds)} / {n_before}")

Map (num_proc=4):   0%|          | 0/913277 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3044258 > 131072). Running this sequence through the model will result in indexing errors


Filter (num_proc=4):   0%|          | 0/913277 [00:00<?, ? examples/s]

После фильтра token_len <= 1024: 719876 / 913277


In [10]:
# Шаг 3: MD5-дедупликация.

def msg_hash(messages):
    joined = "\n".join(f"{m['role']}:{m['content']}" for m in messages)
    return hashlib.md5(joined.encode("utf-8")).hexdigest()

seen = set()
keep_idx = []
for i, messages in enumerate(ds["messages"]):
    h = msg_hash(messages)
    if h not in seen:
        seen.add(h)
        keep_idx.append(i)

n_before = len(ds)
ds = ds.select(keep_idx)
print(f"После дедупликации: {len(ds)} / {n_before}")

После дедупликации: 709908 / 719876


In [11]:
# Шаг 4: stratified sampling 50K по source.
# Каждому источнику — целевая доля = его доля в отфильтрованном ds.
# Но фиксированный минимум на источник (чтобы не терять редкие источники).

MIN_PER_SOURCE = 50

sources = list(ds["source"])
source_to_indices = defaultdict(list)
for i, s in enumerate(sources):
    source_to_indices[s].append(i)

total = len(ds)
selected = []
rng = random.Random(SEED)

for src, idx_list in source_to_indices.items():
    frac = len(idx_list) / total
    quota = max(MIN_PER_SOURCE, int(round(frac * TARGET_SIZE)))
    quota = min(quota, len(idx_list))
    selected.extend(rng.sample(idx_list, quota))

# Округления могли слегка превысить/недобрать TARGET_SIZE — подстраиваем.
rng.shuffle(selected)
if len(selected) > TARGET_SIZE:
    selected = selected[:TARGET_SIZE]
elif len(selected) < TARGET_SIZE:
    remaining = list(set(range(total)) - set(selected))
    selected.extend(rng.sample(remaining, TARGET_SIZE - len(selected)))

assert len(selected) == TARGET_SIZE
ds_sampled = ds.select(selected)
print("Итоговый размер:", len(ds_sampled))
print("\nРаспределение по источникам после сэмплинга:")
for src, cnt in Counter(ds_sampled["source"]).most_common():
    print(f"  {cnt:>5}  {src}")

Итоговый размер: 50000

Распределение по источникам после сэмплинга:
   7141  ai2-adapt-dev/evol_codealpaca_heval_decontaminated
   6134  ai2-adapt-dev/tulu_v3.9_aya_100k
   6099  ai2-adapt-dev/flan_v2_converted
   3737  ai2-adapt-dev/numinamath_tir_math_decontaminated
   3518  ai2-adapt-dev/tulu_v3.9_open_math_2_gsm8k_50k
   3506  allenai/tulu-3-sft-personas-math-grade
   3497  ai2-adapt-dev/tulu_v3.9_synthetic_finalresp_wildguardmixtrain_decontaminated_50k
   3331  ai2-adapt-dev/tulu_v3.9_wildchat_100k
   3171  ai2-adapt-dev/tulu_v3.9_wildjailbreak_decontaminated_50k
   2459  ai2-adapt-dev/personahub_code_v2_34999
   2145  ai2-adapt-dev/personahub_math_v5_regen_149960
   2052  ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980
    771  ai2-adapt-dev/coconot_converted
    622  ai2-adapt-dev/tulu_v3.9_personahub_math_interm_algebra_20k
    602  ai2-adapt-dev/no_robots_converted
    474  ai2-adapt-dev/oasst1_converted
    469  ai2-adapt-dev/tulu_v3.9_sciriff_10k
    248  ai2-adapt-dev

## 5. Train / val split

Stratified split по source: 95/5.

In [12]:
by_src = defaultdict(list)
for i, s in enumerate(ds_sampled["source"]):
    by_src[s].append(i)

train_idx, val_idx = [], []
rng = random.Random(SEED)
for src, idxs in by_src.items():
    rng.shuffle(idxs)
    n_val = max(1, int(round(len(idxs) * VAL_FRACTION)))
    val_idx.extend(idxs[:n_val])
    train_idx.extend(idxs[n_val:])

rng.shuffle(train_idx)
rng.shuffle(val_idx)

train_ds = ds_sampled.select(train_idx)
val_ds = ds_sampled.select(val_idx)
print(f"train: {len(train_ds)}  val: {len(val_ds)}")

train: 47499  val: 2501


## 6. Сохранение артефактов

In [13]:
def dump_jsonl(dataset, path):
    with open(path, "w", encoding="utf-8") as f:
        for ex in dataset:
            rec = {"messages": ex["messages"], "source": ex["source"], "token_len": ex["token_len"]}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

train_path = OUT_DIR / "train.jsonl"
val_path = OUT_DIR / "val.jsonl"
dump_jsonl(train_ds, train_path)
dump_jsonl(val_ds, val_path)
print(f"Saved: {train_path} ({train_path.stat().st_size / 1e6:.1f} MB)")
print(f"Saved: {val_path}   ({val_path.stat().st_size / 1e6:.1f} MB)")

Saved: /kaggle/working/train.jsonl (79.8 MB)
Saved: /kaggle/working/val.jsonl   (4.2 MB)


In [14]:
stats = {
    "model": MODEL_NAME,
    "source_dataset": DATA_NAME,
    "seed": SEED,
    "max_seq_len": MAX_SEQ_LEN,
    "max_turns": MAX_TURNS,
    "target_size": TARGET_SIZE,
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "source_distribution": dict(Counter(ds_sampled["source"])),
    "token_len_percentiles": {
        "p50": float(np.percentile(ds_sampled["token_len"], 50)),
        "p75": float(np.percentile(ds_sampled["token_len"], 75)),
        "p90": float(np.percentile(ds_sampled["token_len"], 90)),
        "p95": float(np.percentile(ds_sampled["token_len"], 95)),
        "p99": float(np.percentile(ds_sampled["token_len"], 99)),
        "max": int(max(ds_sampled["token_len"])),
    },
    "chatml_template": CHATML_TEMPLATE,
}

stats_path = OUT_DIR / "dataset_stats.json"
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print("Saved:", stats_path)
print(json.dumps(stats, indent=2, ensure_ascii=False)[:800])

Saved: /kaggle/working/dataset_stats.json
{
  "model": "Qwen/Qwen3-1.7B",
  "source_dataset": "allenai/tulu-3-sft-mixture",
  "seed": 42,
  "max_seq_len": 1024,
  "max_turns": 4,
  "target_size": 50000,
  "train_size": 47499,
  "val_size": 2501,
  "source_distribution": {
    "ai2-adapt-dev/tulu_v3.9_wildjailbreak_decontaminated_50k": 3171,
    "ai2-adapt-dev/flan_v2_converted": 6099,
    "ai2-adapt-dev/evol_codealpaca_heval_decontaminated": 7141,
    "ai2-adapt-dev/tulu_v3.9_aya_100k": 6134,
    "ai2-adapt-dev/tulu_v3.9_synthetic_finalresp_wildguardmixtrain_decontaminated_50k": 3497,
    "ai2-adapt-dev/numinamath_tir_math_decontaminated": 3737,
    "ai2-adapt-dev/tulu_v3.9_open_math_2_gsm8k_50k": 3518,
    "ai2-adapt-dev/personahub_math_v5_regen_149960": 2145,
    "ai2-adapt-dev/personahub_code_v2_34999": 2459,
    "ai2-adapt-dev


## 7. Следующий шаг

В Kaggle UI: `File → Save Version → создать Dataset` → `kotshubin/qwen-fft-data-50k`.
Затем в `2-qwen-fft-sft.ipynb` подключаем этот dataset как input и стартуем FSDP-тренинг.